In [2]:
import os
import json
import shutil
import random
from pathlib import Path
import logging
import torch
from ultralytics import YOLO
import yaml
import numpy as np

In [3]:
def convert_coco_to_yolo(annotation_path: Path, img_dir: Path, label_dir: Path):
    """Convert COCO format annotations to YOLO format"""
    # Create labels directory
    label_dir.mkdir(parents=True, exist_ok=True)

    # Load COCO annotations
    with open(annotation_path) as f:
        coco_data = json.load(f)

    # Create image_id to image info mapping
    image_info = {img["id"]: img for img in coco_data["images"]}

    # Group annotations by image_id
    annotations_by_image = {}
    for ann in coco_data["annotations"]:
        image_id = ann["image_id"]
        if image_id not in annotations_by_image:
            annotations_by_image[image_id] = []
        annotations_by_image[image_id].append(ann)

    # Process each image
    for image_id, annotations in annotations_by_image.items():
        image = image_info[image_id]
        img_width = image["width"]
        img_height = image["height"]

        # Create YOLO format label file
        label_file = label_dir / Path(image["file_name"]).with_suffix(".txt").name

        with open(label_file, "w") as f:
            for ann in annotations:
                # Get segmentation polygons
                polygons = ann["segmentation"]
                # For each polygon in the annotation
                for polygon in polygons:
                    # Convert to normalized coordinates
                    normalized = []
                    for i in range(0, len(polygon), 2):
                        x = polygon[i] / img_width
                        y = polygon[i + 1] / img_height
                        normalized.extend([x, y])

                    # Write to file: class_id x1 y1 x2 y2 ...
                    f.write(f"0 {' '.join(map(str, normalized))}\n")

In [4]:
# Configure logging
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)


def get_device():
    if torch.cuda.is_available():
        device = 0
        logging.info(f"Using GPU: {torch.cuda.get_device_name(0)}")
    else:
        device = "cpu"
        logging.info("No GPU available, using CPU")
    return device

In [5]:
def create_yaml_config(base_path: Path):
    """Create YAML configuration file for training"""
    """
    config = {
        'path': str(base_path),
        'train': str(base_path / 'train'),  # Directory containing images and labels
        'val': str(base_path / 'val'),  # Directory containing images and labels
        'test': str(base_path / 'test'),  # Directory containing images and labels
        'names': {0: 'peanut'},
        'nc': 1
    }
    """
    config = {
        "train": "train",
        "val": "val",
        "test": "test",
        "names": {0: "peanut"},
        "nc": 1,
    }

    print(config)
    yaml_path = base_path / "dataset.yaml"
    with open(yaml_path, "w") as f:
        yaml.safe_dump(config, f, default_flow_style=False)

    logging.info("Created dataset.yaml configuration file")
    return yaml_path

In [6]:
def process_dataset(source_dir: Path, destination_dir: Path):
    test_dir = destination_dir / "test"

    # If test directory exists, skip processing
    if test_dir.exists():
        logging.info("Using existing dataset split")
        return destination_dir / "dataset.yaml"

    logging.info("Creating new dataset split...")

    # Create directories
    for split in ["train", "val", "test"]:
        split_dir = destination_dir / split
        split_dir.mkdir(parents=True, exist_ok=True)

    # Load COCO annotations
    with open(source_dir / "annotation_coco.json", "r") as f:
        coco_data = json.load(f)

    # Get all image IDs and shuffle
    image_ids = [img["id"] for img in coco_data["images"]]
    random.shuffle(image_ids)

    # Split into train/val/test
    total = len(image_ids)
    train_size = int(0.7 * total)
    val_size = int(0.2 * total)

    splits = {
        "train": image_ids[:train_size],
        "val": image_ids[train_size : train_size + val_size],
        "test": image_ids[train_size + val_size :],
    }

    # Process each split
    for split_name, split_ids in splits.items():
        # Create new COCO structure
        split_data = {
            "info": coco_data["info"],
            "licenses": coco_data["licenses"],
            "categories": coco_data["categories"],
            "images": [],
            "annotations": [],
        }

        # Filter images and annotations
        id_set = set(split_ids)
        split_data["images"] = [
            img for img in coco_data["images"] if img["id"] in id_set
        ]
        split_data["annotations"] = [
            anno for anno in coco_data["annotations"] if anno["image_id"] in id_set
        ]

        # Copy images and save annotations
        split_dir = destination_dir / split_name
        img_dir = split_dir
        img_dir.mkdir(parents=True, exist_ok=True)
        for img in split_data["images"]:
            shutil.copy2(source_dir / img["file_name"], img_dir / img["file_name"])

        # Save COCO annotations
        annotation_path = split_dir / "annotation_coco.json"
        with open(annotation_path, "w") as f:
            json.dump(split_data, f)

        # Convert to YOLO format
        label_dir = split_dir
        label_dir.mkdir(parents=True, exist_ok=True)
        convert_coco_to_yolo(annotation_path, img_dir, label_dir)

        logging.info(f"Created {split_name} split with {len(split_ids)} images")

    # Create YAML configuration file
    return create_yaml_config(destination_dir)

In [ ]:
import gc

if __name__ == "__main__":

    # Process dataset and get yaml path
    yaml_path = process_dataset(
        Path(r"Dataset/Images").resolve(), Path(r"Dataset").resolve()
    )

    gc.collect()
    torch.cuda.empty_cache()

    # Set memory allocator config
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

    # Train YOLOv8 model
    model = YOLO(Path(r"yolov8s-seg.pt"))

    model.train(
        data=str(yaml_path),
        epochs=100,
        imgsz=1024,
        batch=4,
        device=get_device(),
        workers=4,
        task="segment",
    )
    logging.info("Training completed successfully")

2024-11-06 08:46:44,108 - INFO - Using existing dataset split
2024-11-06 08:46:45,337 - INFO - Using GPU: Tesla P100-SXM2-16GB


Ultralytics 8.3.27 🚀 Python-3.11.9 torch-2.4.0 CUDA:0 (Tesla P100-SXM2-16GB, 16276MiB)
engine/trainer: task=segment, mode=train, model=yolov8s-seg.pt, data=/workspace/Dataset/dataset.yaml, epochs=100, time=None, patience=100, batch=4, imgsz=1024, save=True, save_period=-1, cache=False, device=0, workers=4, project=None, name=train13, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=T

train: Scanning /workspace/Dataset/train.cache... 48 images, 0 backgrounds, 0 corrupt: 100%|██████████| 48/48 [00:00<?, ?it/s]
val: Scanning /workspace/Dataset/val.cache... 19 images, 0 backgrounds, 0 corrupt: 100%|██████████| 19/19 [00:00<?, ?it/s]


Plotting labels to runs/segment/train13/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 66 weight(decay=0.0), 77 weight(decay=0.0005), 76 bias(decay=0.0)
Image sizes 1024 train, 1024 val
Using 4 dataloader workers
Logging results to runs/segment/train13
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss  Instances       Size


      1/100      12.7G       1.04      2.389      3.267      1.053        952       1024:  58%|█████▊    | 7/12 [00:04<00:03,  1.59it/s]

In [ ]:
torch.cuda.empty_cache()
gc.collect()

In [ ]:
print(torch.cuda.memory_summary())